# Part 2: Flow Matching on MNIST

> **Learning objectives:**
> - Understand how adaptive layer normalization (AdaLN) injects conditioning into a transformer
> - Assemble a Diffusion Transformer (DiT) from provided building blocks
> - Adapt flow matching loss and sampling from Part 1 to work with images
> - Train a class-conditional image generator with classifier-free guidance

**Prerequisites:** Part 1 (flow matching basics). You should understand flow matching loss, Euler sampling, and CFG.

In [ ]:
import torch
import torch as t
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import sys
from pathlib import Path

exercises_dir = Path(".").resolve().parent
if str(exercises_dir) not in sys.path:
    sys.path.insert(0, str(exercises_dir))

from part2_flow_matching_mnist import solutions
from part2_flow_matching_mnist.tests import (
    test_patch, test_unpatch, test_dit_block, test_dit,
    test_sample,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Provided Building Blocks

The following components are provided — read through them to understand how they work, but you don't need to implement them.

- **`RMSNorm`**: Root mean square normalization
- **`Attention`**: Multi-head self-attention with QK-normalization (bidirectional, no causal mask)
- **`MLP`**: Gated MLP with SiLU activation
- **`NumEmbedding`**: Sinusoidal embeddings for timestep conditioning

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d=32, eps=1e-6):
        super().__init__()
        self.scale = nn.Parameter(t.ones((1, 1, d)))
        self.eps = eps

    def forward(self, x):
        rms = (x.pow(2).mean(dim=-1, keepdim=True)).sqrt() + self.eps
        return x / rms * self.scale


class Attention(nn.Module):
    def __init__(self, d=32, n_head=4):
        super().__init__()
        self.n_head = n_head
        self.d = d
        self.d_head = d // n_head
        self.QKV = nn.Linear(d, 3 * d, bias=False)
        self.O = nn.Linear(d, d, bias=False)
        self.normq = RMSNorm(self.d_head)
        self.normk = RMSNorm(self.d_head)

    def forward(self, x):
        b, s, d = x.shape
        q, k, v = self.QKV(x).chunk(3, dim=-1)
        q = q.reshape(b, s, self.n_head, self.d_head)
        k = k.reshape(b, s, self.n_head, self.d_head)
        v = v.reshape(b, s, self.n_head, self.d_head)
        q = self.normq(q)
        k = self.normk(k)
        attn = q.permute(0, 2, 1, 3) @ k.permute(0, 2, 3, 1)
        attn = attn.softmax(dim=-1)
        z = attn @ v.permute(0, 2, 1, 3)
        z = z.permute(0, 2, 1, 3).reshape(b, s, self.d)
        return self.O(z)


class MLP(nn.Module):
    def __init__(self, d=32, exp=2):
        super().__init__()
        self.up = nn.Linear(d, exp * d, bias=False)
        self.gate = nn.Linear(d, exp * d, bias=False)
        self.down = nn.Linear(exp * d, d, bias=False)
        self.act = nn.SiLU()

    def forward(self, x):
        return self.down(self.up(x) * self.act(self.gate(x)))


class NumEmbedding(nn.Module):
    """Sinusoidal embeddings for integer values (timesteps)."""
    def __init__(self, n_max, d=32, C=500):
        super().__init__()
        thetas = C ** (-t.arange(0, d // 2) / (d // 2))
        thetas = t.arange(0, n_max)[:, None].float() @ thetas[None, :]
        sins = t.sin(thetas)
        coss = t.cos(thetas)
        self.register_buffer("E", t.cat([sins, coss], dim=1))

    def forward(self, x):
        return self.E[x]

## Exercise 1 — Patch (Image → Tokens)
*Difficulty: 🔴🔴⭕⭕⭕ | Importance: 🔵🔵🔵🔵⭕ | ~10 min*

Convert images into patch token sequences. An image of shape `(B, C, H, W)` with patch size `P` becomes `(B, (H/P)*(W/P), d)` tokens.

Use a strided convolution (`nn.Conv2d` with `stride=patch_size`) to both extract and project patches in one step.

In [ ]:
class Patch(nn.Module):
    """Convert images to patch token sequences using strided convolution."""
    def __init__(self, patch_size=4, in_channels=1, d=32):
        super().__init__()
        self.d = d
        self.patch_size = patch_size
        self.conv = nn.Conv2d(in_channels, d, kernel_size=5, padding=2, stride=patch_size)

    def forward(self, x):
        b, c, h, w = x.shape
        x = self.conv(x)                           # (b, d, h//ps, w//ps)
        x = x.permute(0, 2, 3, 1)                  # (b, h//ps, w//ps, d)
        x = x.reshape(b, -1, self.d)               # (b, n_patches, d)
        return x


In [ ]:
test_patch(Patch)

## Exercise 2 — UnPatch (Tokens → Image)
*Difficulty: 🔴🔴⭕⭕⭕ | Importance: 🔵🔵🔵🔵⭕ | ~10 min*

Reverse the patchification: project each token to `patch_size^2 * channels` values and rearrange back into an image.

Hint: `h = w = int(s ** 0.5)` where `s` is the number of patches.

In [ ]:
class UnPatch(nn.Module):
    """Convert patch token sequences back to images."""
    def __init__(self, patch_size=4, d=32, out_channels=1):
        super().__init__()
        self.out_channels = out_channels
        self.patch_size = patch_size
        self.up = nn.Linear(d, patch_size ** 2 * out_channels)

    def forward(self, x):
        b, s, d = x.shape
        x = self.up(x)
        w = int(s ** 0.5)
        h = w
        ps = self.patch_size
        c = self.out_channels
        x = x.reshape(b, h, w, c, ps, ps)
        x = x.permute(0, 3, 1, 4, 2, 5)
        x = x.reshape(b, c, h * ps, w * ps)
        return x


In [ ]:
test_unpatch(Patch, UnPatch)

## Exercise 3 — DiT Block with Adaptive Layer Normalization
*Difficulty: 🔴🔴🔴🔴⭕ | Importance: 🔵🔵🔵🔵🔵 | ~25 min*

This is the key architectural idea in Diffusion Transformers. Instead of a standard transformer block where normalization has fixed learned parameters, we **modulate** the normalization using the conditioning signal (timestep + class).

The conditioning vector `c` (shape `(B, d)`) produces **6 modulation parameters** via a linear layer:
- `scale1, bias1, gate1` for the attention sub-layer
- `scale2, bias2, gate2` for the MLP sub-layer

The modulated forward pass for each sub-layer is:

$$x \leftarrow x + \text{gate} \odot \text{sublayer}(\text{norm}(x) \odot (1 + \text{scale}) + \text{bias})$$

Note: `c` is `(B, d)` but `x` is `(B, seq, d)`, so you need to broadcast with `[:, None, :]`.

In [ ]:
from IPython.display import display, HTML

display(HTML('''
<div style="background: #2b2b2b; padding: 20px; border-radius: 8px; display: inline-block;">
  <img src="dit_block.svg" width="400"/>
</div>
'''))

In [ ]:
class DiTBlock(nn.Module):
    def __init__(self, d=32, n_head=4, exp=2):
        super().__init__()
        self.norm1 = RMSNorm(d)
        self.attn = Attention(d, n_head)
        self.norm2 = RMSNorm(d)
        self.mlp = MLP(d, exp)
        self.modulate = nn.Linear(d, 6 * d)

    def forward(self, x, c):
        scale1, bias1, gate1, scale2, bias2, gate2 = self.modulate(c).chunk(6, dim=-1)

        residual = x
        x = self.norm1(x) * (1 + scale1[:, None, :]) + bias1[:, None, :]
        x = self.attn(x) * gate1[:, None, :]
        x = residual + x

        residual = x
        x = self.norm2(x) * (1 + scale2[:, None, :]) + bias2[:, None, :]
        x = self.mlp(x) * gate2[:, None, :]
        x = residual + x
        return x


In [ ]:
test_dit_block(DiTBlock)

## Exercise 4 — Full DiT Model
*Difficulty: 🔴🔴🔴⭕⭕ | Importance: 🔵🔵🔵🔵🔵 | ~20 min*

Assemble the full Diffusion Transformer:

1. **Convert timestep** `ts` (float in [0,1]) to integer indices, look up sinusoidal embedding
2. **Build conditioning**: `cond = SiLU(time_emb + class_emb)`
3. **Patchify** input image + add learned position embeddings
4. **Stack of DiTBlocks** with conditioning
5. **Final modulated norm** (2 params: scale, bias) + unpatch back to image

In [ ]:
display(HTML('''
<div style="background: #2b2b2b; padding: 20px; border-radius: 8px; display: inline-block;">
  <img src="dit.svg" width="400"/>
</div>
'''))

In [ ]:
class DiT(nn.Module):
    def __init__(self, h=28, w=28, n_classes=11, in_channels=1,
                 patch_size=2, n_blocks=6, d=192, n_head=12, exp=2, T=1000):
        super().__init__()
        self.T = T
        self.patch = Patch(patch_size, in_channels, d)
        self.n_seq = (h // patch_size) * (w // patch_size)
        self.pe = nn.Parameter(t.randn(1, self.n_seq, d) * d ** -0.5)
        self.te = NumEmbedding(T, d)
        self.ce = nn.Embedding(n_classes, d)
        self.act = nn.SiLU()
        self.blocks = nn.ModuleList([DiTBlock(d, n_head, exp) for _ in range(n_blocks)])
        self.norm = RMSNorm(d)
        self.modulate = nn.Linear(d, 2 * d)
        self.unpatch = UnPatch(patch_size, d, in_channels)

    def forward(self, x, c, ts):
        ts_int = t.minimum((ts * self.T).to(t.int64), t.tensor(self.T - 1, device=ts.device))
        cond = self.act(self.te(ts_int) + self.ce(c))
        x = self.patch(x) + self.pe
        for block in self.blocks:
            x = block(x, cond)
        scale, bias = self.modulate(cond).chunk(2, dim=-1)
        x = self.norm(x) * (1 + scale[:, None, :]) + bias[:, None, :]
        x = self.unpatch(x)
        return x

    @property
    def device(self):
        return self.pe.device

    @property
    def dtype(self):
        return self.pe.dtype


In [ ]:
test_dit(DiT)

## Optimizer: Muon

Before training, we set up the [Muon optimizer](https://github.com/KellerJordan/Muon). Muon uses momentum-based updates for large weight matrices and falls back to Adam for everything else (biases, gains, embeddings).

**The key idea:** split parameters into two groups based on dimensionality:
- **`ndim >= 2`** (weight matrices in transformer blocks) → Muon with `lr1=0.02`
- **`ndim < 2`** (biases, layer norm gains, embeddings) → Adam with `lr2=3e-4`

We also separate **block parameters** (inside `model.blocks`) from **other parameters** (patch/unpatch embeddings, final layers). Only block weight matrices use Muon.

In [ ]:
from muon import SingleDeviceMuonWithAuxAdam

def get_muon(model, lr1=0.02, lr2=3e-4, betas=(0.9, 0.95), weight_decay=1e-5):
    """Create Muon optimizer with param group splitting.

    Args:
        model: nn.Module with a .blocks attribute
        lr1: learning rate for hidden weight matrices (Muon)
        lr2: learning rate for biases/gains/embeddings (Adam)
    """
    # Separate block params from everything else
    body_weights = list(model.blocks.parameters())
    body_ids = {id(p) for p in body_weights}
    other_weights = [p for p in model.parameters() if id(p) not in body_ids]

    # Within blocks: weight matrices (ndim >= 2) vs biases/gains (ndim < 2)
    hidden_weights = [p for p in body_weights if p.ndim >= 2]
    hidden_gains_biases = [p for p in body_weights if p.ndim < 2]

    param_groups = [
        dict(params=hidden_weights, use_muon=True, lr=lr1, weight_decay=weight_decay),
        dict(params=hidden_gains_biases + other_weights, use_muon=False,
             lr=lr2, betas=betas, weight_decay=weight_decay),
    ]
    return SingleDeviceMuonWithAuxAdam(param_groups)

# Verify the split
model_test = DiT(h=28, w=28, n_classes=11)
body = list(model_test.blocks.parameters())
print(f"Block params: {len(body)} tensors")
print(f"  ndim >= 2 (Muon): {sum(1 for p in body if p.ndim >= 2)} tensors, {sum(p.numel() for p in body if p.ndim >= 2):,} params")
print(f"  ndim < 2 (Adam):  {sum(1 for p in body if p.ndim < 2)} tensors, {sum(p.numel() for p in body if p.ndim < 2):,} params")
other = [p for p in model_test.parameters() if id(p) not in {id(p) for p in body}]
print(f"Other params (Adam): {len(other)} tensors, {sum(p.numel() for p in other):,} params")

## Exercise 5 — Training
*Difficulty: 🔴🔴🔴⭕⭕ | Importance: 🔵🔵🔵🔵🔵 | ~15 min*

Train the DiT with flow matching loss inline and logit-normal timestep sampling (`t = sigmoid(randn())`).

Use class labels with 20% label dropout for CFG. Shift labels by +1 (0 = unconditional).


In [ ]:
# Load MNIST
from torchvision import datasets, transforms
train_dataset = datasets.MNIST(root='./data', train=True, download=True,
                                transform=transforms.ToTensor())
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)


In [ ]:
model = DiT(h=28, w=28, n_classes=11).to(device)
optimizer = get_muon(model, lr1=0.02, lr2=3e-4)
label_dropout = 0.2

for epoch in range(3):
    model.train()
    epoch_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        c = labels + 1  # shift: 0 = unconditional
        drop_mask = t.rand(c.shape[0], device=device) < label_dropout
        c[drop_mask] = 0

        ts = F.sigmoid(t.randn(images.shape[0], device=images.device))  # logit-normal
        z = t.randn_like(images)
        v_true = images - z
        x_t = images - ts[:, None, None, None] * v_true
        v_pred = model(x_t, c, ts)
        loss = F.mse_loss(v_pred, v_true)

        optimizer.zero_grad()
        loss.backward()
        t.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        optimizer.step()
        epoch_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {epoch_loss / len(train_loader):.4f}")


## Exercise 6 — Euler Sampling for Images
*Difficulty: 🔴🔴⭕⭕⭕ | Importance: 🔵🔵🔵🔵🔵 | ~10 min*

Same Euler integration as Part 1, using the SD3 schedule. The model takes `(z, y, t_batch)`. Support optional CFG.


In [ ]:
@t.no_grad()
def sample(model, z, y, n_steps=30, cfg=0):
    """Generate images from noise using Euler integration."""
    ts = t.linspace(1, 0, n_steps + 1, device=z.device, dtype=z.dtype)
    ts = 3 * ts / (2 * ts + 1)  # SD3 scheduler
    for idx in range(n_steps):
        t_batch = ts[idx] * t.ones(z.shape[0], dtype=z.dtype, device=z.device)
        v_pred = model(z, y, t_batch)
        if cfg > 0:
            v_uncond = model(z, y * 0, t_batch)
            v_pred = v_uncond + cfg * (v_pred - v_uncond)
        z = z + (ts[idx] - ts[idx + 1]) * v_pred
    return z


In [ ]:
test_sample(sample, DiT)

In [ ]:
model.eval()

# Generate conditional samples (2 per digit, CFG=3)
z = t.randn(20, 1, 28, 28, device=device)
y_cond = (t.arange(10, device=device).repeat(2) + 1)  # shifted labels
samples_cond = sample(model, z, y_cond, n_steps=30, cfg=3)

# Generate unconditional samples
z2 = t.randn(20, 1, 28, 28, device=device)
y_uncond = t.zeros(20, dtype=t.long, device=device)
samples_uncond = sample(model, z2, y_uncond, n_steps=30, cfg=0)

# Collect 2 real examples per digit, but in column-major order:
# 0 1 2 3 4 ... 9 | 0 1 2 3 4 ... 9
real_images_per_digit = [[] for _ in range(10)]
for img, label in train_dataset:
    if len(real_images_per_digit[label]) < 2:
        real_images_per_digit[label].append(img)
    if all(len(lst) == 2 for lst in real_images_per_digit):
        break
# Now interleave as 0,1,2,...,9,0,1,2,...,9 for 20 entries
real_images = []
for j in range(2):           # for each shot
    for i in range(10):      # for each digit
        real_images.append(real_images_per_digit[i][j])
real_images = t.stack(real_images)

# Plot: 6 rows x 10 cols
fig, axes = plt.subplots(6, 10, figsize=(15, 9))
for i in range(10):
    for j in range(2):
        idx = j * 10 + i  # Now the index properly selects row j, col i as digit i & shot j
        axes[j, i].imshow(real_images[idx, 0].cpu(), cmap='gray')
        axes[j, i].axis('off')
        if j == 0:
            axes[j, i].set_title(str(i), fontsize=12)
        axes[j + 2, i].imshow(samples_cond[idx, 0].cpu().clamp(0, 1), cmap='gray')
        axes[j + 2, i].axis('off')
        axes[j + 4, i].imshow(samples_uncond[idx, 0].cpu().clamp(0, 1), cmap='gray')
        axes[j + 4, i].axis('off')

for ax, label in zip([axes[0, 0], axes[2, 0], axes[4, 0]], ['Real', 'Cond\n(CFG=3)', 'Uncond']):
    ax.annotate(label, xy=(-0.3, 0.5), xycoords='axes fraction',
                fontsize=11, fontweight='bold', ha='right', va='center')

plt.tight_layout()
plt.show()


## Summary

You built a complete image generation pipeline:
- **AdaLN modulation** is how diffusion transformers inject conditioning — the key architectural insight
- **DiT** combines patchification, positional embeddings, modulated transformer blocks, and unpatchification
- **Flow matching loss** and **Euler sampling** extend directly from Part 1 to higher dimensions
- **CFG** with label dropout enables controllable class-conditional generation

Next: **Part 3** extends this to video with causal attention, action conditioning, and diffusion forcing.